In [1]:
import requests
import pandas as pd
import json

In [2]:
# Open Charge Map API key
API_KEY = "f82b0388-2e7e-4d18-8207-6fe535c64cb8"

# Open Charge Map API endpoint for charging-station data
url = "https://api.openchargemap.io/v3/poi/"

# Parameters used to request charging stations located in India
params = {
    "output": "json",
    "countrycode": "IN",
    "maxresults": 5000,
    "compact": False,
    "verbose": False,
    "key": API_KEY
}

# Send the request to the Open Charge Map API
response = requests.get(url, params=params)
print("Status code:", response.status_code)

Status code: 200


In [3]:
# Convert the API response from JSON format into Python objects
data = response.json()

# Check how many charging-station records were retrieved
print("Number of charging stations pulled:", len(data))

Number of charging stations pulled: 1979


In [4]:
# Inspect one raw API record to understand its nested structure
data[0]

{'DataProvider': {'WebsiteURL': 'http://openchargemap.org',
  'DataProviderStatusType': {'IsProviderEnabled': True,
   'ID': 1,
   'Title': 'Manual Data Entry'},
  'IsRestrictedEdit': False,
  'IsOpenDataLicensed': True,
  'IsApprovedImport': True,
  'License': 'Licensed under Creative Commons Attribution 4.0 International (CC BY 4.0)',
  'ID': 1,
  'Title': 'Open Charge Map Contributors'},
 'OperatorInfo': {'WebsiteURL': 'https://efillelectric.com/',
  'IsPrivateIndividual': False,
  'IsRestrictedEdit': False,
  'ID': 3989,
  'Title': 'E-Fill Electric (IN)'},
 'UsageType': {'IsPayAtLocation': True,
  'IsMembershipRequired': False,
  'IsAccessKeyRequired': False,
  'ID': 5,
  'Title': 'Public - Pay At Location'},
 'StatusType': {'IsOperational': True,
  'IsUserSelectable': True,
  'ID': 50,
  'Title': 'Operational'},
 'SubmissionStatus': {'IsLive': True,
  'ID': 200,
  'Title': 'Submission Published'},
 'UserComments': [{'ID': 32125,
   'ChargePointID': 510840,
   'CommentTypeID': 10,


In [5]:
# Flatten everything except Connections automatically
df_chargers = pd.json_normalize(data)

# Check the number of rows and columns after flattening
print("Shape after normalize:", df_chargers.shape)

# Display all available columns so we can decide
# which fields are useful for the project
print("\nAll column names:")
print(df_chargers.columns.tolist())

Shape after normalize: (1979, 72)

All column names:
['UserComments', 'MediaItems', 'IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID', 'DataProviderID', 'OperatorID', 'UsageTypeID', 'UsageCost', 'Connections', 'NumberOfPoints', 'GeneralComments', 'StatusTypeID', 'DateLastStatusUpdate', 'DataQualityLevel', 'DateCreated', 'SubmissionStatusTypeID', 'DataProvider.WebsiteURL', 'DataProvider.DataProviderStatusType.IsProviderEnabled', 'DataProvider.DataProviderStatusType.ID', 'DataProvider.DataProviderStatusType.Title', 'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed', 'DataProvider.IsApprovedImport', 'DataProvider.License', 'DataProvider.ID', 'DataProvider.Title', 'OperatorInfo.WebsiteURL', 'OperatorInfo.IsPrivateIndividual', 'OperatorInfo.IsRestrictedEdit', 'OperatorInfo.ID', 'OperatorInfo.Title', 'UsageType.IsPayAtLocation', 'UsageType.IsMembershipRequired', 'UsageType.IsAccessKeyRequired', 'UsageType.ID', 'UsageType.Title', 'StatusType.IsOperational', 'StatusType.Is

In [6]:
# Rename long API column names into shorter and easier-to-use names for the cleaning and analysis stages.
df_chargers = df_chargers.rename(columns={
    "AddressInfo.Title": "name",
    "AddressInfo.Latitude": "latitude",
    "AddressInfo.Longitude": "longitude",
    "AddressInfo.Town": "town",
    "AddressInfo.StateOrProvince": "state",
    "AddressInfo.Postcode": "postcode",
    "OperatorInfo.Title": "operator",
    "StatusType.Title": "status",
    "StatusType.IsOperational": "is_operational",
    "DateCreated": "date_created",
    "DateLastVerified": "date_last_verified"
})

In [7]:
# Find the highest charging power available at each station.
def get_max_power(connections):
    # Return None when connection information is missing
    if not connections or not isinstance(connections, list):
        return None

    # Collect valid PowerKW values from all connectors
    powers = [
        c.get("PowerKW")
        for c in connections
        if c.get("PowerKW")
    ]

    # Return the highest power value available
    return max(powers) if powers else None


In [8]:
# Calculate the total number of charging points
# represented by all connections at a station.
def get_total_points(connections):
    # Return 0 when connection information is unavailable
    if not connections or not isinstance(connections, list):
        return 0

    # Sum the Quantity value from each connection.
    # If Quantity is missing, assume one charging point.
    return sum(c.get("Quantity", 1) or 1 for c in connections)


In [9]:
# Check whether at least one connection supports
# fast charging.
def has_fast_charging(connections):
    # Return False when no connection information exists
    if not connections or not isinstance(connections, list):
        return False

    # Return True if any connection is identified
    # as fast-charge capable
    return any(
        c.get("Level", {}).get("IsFastChargeCapable", False)
        for c in connections
    )

In [10]:
# Apply the functions to the nested Connections column
df_chargers["max_power_kw"] = (
    df_chargers["Connections"].apply(get_max_power)
)

df_chargers["total_points"] = (
    df_chargers["Connections"].apply(get_total_points)
)

df_chargers["has_fast_charging"] = (
    df_chargers["Connections"].apply(has_fast_charging)
)

In [11]:
# Keep only the fields required for the EV charging
# infrastructure analysis.
data = df_chargers[[
    "ID",
    "name",
    "latitude",
    "longitude",
    "town",
    "state",
    "postcode",
    "operator",
    "status",
    "is_operational",
    "max_power_kw",
    "total_points",
    "has_fast_charging",
    "date_created",
    "date_last_verified"
]]

# Confirm the final raw dataset structure
print("Final shape:", data.shape)

# Display the first few records for verification
data.head(10)

Final shape: (1979, 15)


,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,510840,E-Fill Electric Hotel Food Mania,31.602844,76.924626,"Balh, Mandi",Himachal Pradesh,175021,E-Fill Electric (IN),Operational,True,30.0,1,True,2026-09-30T16:02:00Z,2026-10-03T16:15:00Z
1,510821,Nilkanth EV Charging Station - ChargeJet,23.759334,53.322467,Ahmedabad,Gujarat,382445,(Unknown Operator),Operational,True,160.0,3,True,2026-09-30T04:52:00Z,2026-10-03T04:57:00Z
2,510820,Nilkanth EV Charging Station - ChargeJet,22.979910,72.633820,Ahemdabad,Gujarat,382445,(Unknown Operator),Operational,True,158.0,4,True,2026-09-30T04:24:00Z,2026-10-03T04:26:00Z
3,510563,bhanu prasad hostal,16.258243,80.323226,Guntar,Andhra Pradesh,534350,(Unknown Operator),Operational,True,120.0,1,True,2026-09-26T04:27:00Z,2026-09-29T04:31:00Z
4,510426,SALZER DEVELOPMENT,19.105321,72.875887,,NaN,,Salzer NexCharge (IN),Not Operational,False,7.0,1,False,2026-09-23T16:17:00Z,2026-10-04T04:18:00Z
5,510425,Salzer Panel Test Charger 1,11.929501,76.937771,,NaN,,Salzer NexCharge (IN),Not Operational,False,30.0,2,True,2026-09-23T16:17:00Z,2026-10-04T04:18:00Z
6,510424,7.4 kW - New Model Testing,11.163261,76.934373,,NaN,,Salzer NexCharge (IN),Not Operational,False,7.0,1,False,2026-09-23T16:17:00Z,2026-10-04T04:18:00Z
7,510423,Salzer Test Charger,10.589207,79.589475,,NaN,,Salzer NexCharge (IN),Not Operational,False,22.0,1,False,2026-09-23T16:17:00Z,2026-10-04T04:18:00Z
8,510422,Salzer Panel Test Charger 2,10.585263,77.253175,,NaN,,Salzer NexCharge (IN),Not Operational,False,30.0,2,True,2026-09-23T16:17:00Z,2026-10-04T04:18:00Z
9,510421,AN Energy Solution,10.048785,78.349298,,NaN,,Salzer NexCharge (IN),Not Operational,False,3.0,1,False,2026-09-23T16:17:00Z,2026-10-04T04:18:00Z


In [12]:
# Save the API output after basic extraction to the Raw datasets folder.

data.to_csv(
    "../data/Raw datasets/ev_charging_stations.csv",
    index=False
)

print("Saved successfully! Check your /data folder.")

Saved successfully! Check your /data folder.


In [13]:
# Load the raw charging-station dataset collected from Open Charge Map.
df_chargers = pd.read_csv("../data/Raw datasets/ev_charging_stations.csv")

print("Chargers shape:", df_chargers.shape)
df_chargers.head()

Chargers shape: (1979, 15)


,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,510840,E-Fill Electric Hotel Food Mania,31.602844,76.924626,"Balh, Mandi",Himachal Pradesh,175021,E-Fill Electric (IN),Operational,True,30.0,1,True,2026-09-30T16:02:00Z,2026-10-03T16:15:00Z
1,510821,Nilkanth EV Charging Station - ChargeJet,23.759334,53.322467,Ahmedabad,Gujarat,382445,(Unknown Operator),Operational,True,160.0,3,True,2026-09-30T04:52:00Z,2026-10-03T04:57:00Z
2,510820,Nilkanth EV Charging Station - ChargeJet,22.979910,72.633820,Ahemdabad,Gujarat,382445,(Unknown Operator),Operational,True,158.0,4,True,2026-09-30T04:24:00Z,2026-10-03T04:26:00Z
3,510563,bhanu prasad hostal,16.258243,80.323226,Guntar,Andhra Pradesh,534350,(Unknown Operator),Operational,True,120.0,1,True,2026-09-26T04:27:00Z,2026-09-29T04:31:00Z
4,510426,SALZER DEVELOPMENT,19.105321,72.875887,NaN,NaN,NaN,Salzer NexCharge (IN),Not Operational,False,7.0,1,False,2026-09-23T16:17:00Z,2026-10-04T04:18:00Z


In [14]:
print("Column data types:")
print(df_chargers.dtypes)

print("\nMissing values per column:")
print(df_chargers.isnull().sum())

print("\nTotal duplicate rows (exact full-row duplicates):")
print(df_chargers.duplicated().sum())

Column data types:
ID                      int64
name                      str
latitude              float64
longitude             float64
town                      str
state                     str
postcode                  str
operator                  str
status                    str
is_operational           bool
max_power_kw          float64
total_points            int64
has_fast_charging        bool
date_created              str
date_last_verified        str
dtype: object

Missing values per column:
ID                      0
name                    0
latitude                0
longitude               0
town                  441
state                  36
postcode              473
operator              390
status                  0
is_operational          0
max_power_kw          108
total_points            0
has_fast_charging       0
date_created            0
date_last_verified      0
dtype: int64

Total duplicate rows (exact full-row duplicates):
0


In [15]:
df_chargers.info()

<class 'pandas.DataFrame'>
RangeIndex: 1979 entries, 0 to 1978
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  1979 non-null   int64  
 1   name                1979 non-null   str    
 2   latitude            1979 non-null   float64
 3   longitude           1979 non-null   float64
 4   town                1538 non-null   str    
 5   state               1943 non-null   str    
 6   postcode            1506 non-null   str    
 7   operator            1589 non-null   str    
 8   status              1979 non-null   str    
 9   is_operational      1979 non-null   bool   
 10  max_power_kw        1871 non-null   float64
 11  total_points        1979 non-null   int64  
 12  has_fast_charging   1979 non-null   bool   
 13  date_created        1979 non-null   str    
 14  date_last_verified  1979 non-null   str    
dtypes: bool(2), float64(3), int64(2), str(8)
memory usage: 411.7 KB


In [16]:
print("Unique state values BEFORE cleaning:")
print(sorted(df_chargers["state"].dropna().unique()))

Unique state values BEFORE cleaning:
[' Tamil Nadu', 'Ahmedabad', 'Andhra Pradesh', 'Assam', 'Bangalore Urban', 'Bihar', 'Burari', 'Dadra and Nagar Haveli', 'Delhi', 'Delhi ', 'GJ', 'Goa', 'Gujarat', 'Gujarat ', 'Haryana', 'Haryana ', 'Himachal Pradesh', 'India', 'Jammu and Kashmir', 'Jharkhand', 'KA', 'KARNATAKA', 'Karnatak', 'Karnataka', 'Karnataka ', 'Keraka', 'Keral', 'Kerala', 'Kerala ', 'Lerala', 'MH', 'MP', 'Madhya Pradesh', 'Maharashtra', 'Maharashtra ', 'Maharshtra', 'Mahrashtra', 'New Dehi', 'Odisha', 'Puducherry', 'Punjab ', 'RJ', 'Rajasthan', 'Rajasthan ', 'TAMILNADU', 'Tamil Nadu', 'Tamil Nadu ', 'Tamil Nasdu', 'Tamil nadu', 'TamilNadu', 'Tamilnadu', 'Tamilnadu ', 'Telangana', 'Telangana ', 'UTTAR PRADESH', 'UTTAR PRADESH ', 'Uttar Pradesh', 'Uttar Pradesh ', 'Uttarakhand', 'Uttarakhnad', 'Villupuram', 'West Bengal', 'chennai', 'karnataka', 'rajasthan', 'uttar pradesh ', 'west bengal']


In [17]:
# Remove leading and trailing spaces from state names.
df_chargers["state"] = (
    df_chargers["state"]
    .astype(str)
    .str.strip()
)

In [18]:
# Remove leading and trailing spaces from state names.
df_chargers["state"] = (
    df_chargers["state"]
    .astype(str)
    .str.strip()
)


# Mapping used to correct common spelling variations,
# abbreviations and incorrectly entered locations.
state_name_map = {

    # Different spellings/capitalization
    "KARNATAKA": "Karnataka",
    "karnataka": "Karnataka",
    "Karnatak": "Karnataka",

    "TAMILNADU": "Tamil Nadu",
    "TamilNadu": "Tamil Nadu",
    "Tamilnadu": "Tamil Nadu",
    "Tamil nadu": "Tamil Nadu",

    "UTTAR PRADESH": "Uttar Pradesh",
    "uttar pradesh": "Uttar Pradesh",

    "rajasthan": "Rajasthan",
    "west bengal": "West Bengal",

    # State abbreviations
    "GJ": "Gujarat",
    "KA": "Karnataka",
    "MH": "Maharashtra",
    "MP": "Madhya Pradesh",
    "RJ": "Rajasthan",

    # Spelling errors
    "Maharshtra": "Maharashtra",
    "Mahrashtra": "Maharashtra",
    "Keraka": "Kerala",
    "Keral": "Kerala",
    "Lerala": "Kerala",
    "Tamil Nasdu": "Tamil Nadu",
    "Uttarakhnad": "Uttarakhand",
    "New Dehi": "Delhi",

    # Cities/districts incorrectly entered in the state field
    "Ahmedabad": "Gujarat",
    "Bangalore Urban": "Karnataka",
    "Burari": "Delhi",
    "Villupuram": "Tamil Nadu",
    "chennai": "Tamil Nadu",

    # Invalid country-level entry
    "India": None
}


# Apply the state-name corrections
df_chargers["state"] = (
    df_chargers["state"].replace(state_name_map)
)

# Display the result after applying the mapping
df_chargers

,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,510840,E-Fill Electric Hotel Food Mania,31.602844,76.924626,"Balh, Mandi",Himachal Pradesh,175021,E-Fill Electric (IN),Operational,True,30.0,1,True,2026-09-30T16:02:00Z,2026-10-03T16:15:00Z
1,510821,Nilkanth EV Charging Station - ChargeJet,23.759334,53.322467,Ahmedabad,Gujarat,382445,(Unknown Operator),Operational,True,160.0,3,True,2026-09-30T04:52:00Z,2026-10-03T04:57:00Z
2,510820,Nilkanth EV Charging Station - ChargeJet,22.979910,72.633820,Ahemdabad,Gujarat,382445,(Unknown Operator),Operational,True,158.0,4,True,2026-09-30T04:24:00Z,2026-10-03T04:26:00Z
3,510563,bhanu prasad hostal,16.258243,80.323226,Guntar,Andhra Pradesh,534350,(Unknown Operator),Operational,True,120.0,1,True,2026-09-26T04:27:00Z,2026-09-29T04:31:00Z
4,510426,SALZER DEVELOPMENT,19.105321,72.875887,NaN,NaN,NaN,Salzer NexCharge (IN),Not Operational,False,7.0,1,False,2026-09-23T16:17:00Z,2026-10-04T04:18:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1974,103604,"International Tech Park, Bangalore",12.985421,77.738513,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04T08:06:00Z,2018-05-11T11:06:00Z
1975,103603,Ather Grid Point - Ascendas Park square mall,12.987219,77.736474,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04T08:01:00Z,2018-05-11T11:06:00Z
1976,96047,Tellus Power,17.413362,78.426597,Hyderabad,Telangana,500034,NaN,Operational,True,7.2,1,False,2017-12-07T10:12:00Z,2017-12-08T08:52:00Z
1977,92772,Tata Power Receiving Station,19.095796,72.926011,Mumbai,NaN,400077,(Business Owner at Location),Operational,True,NaN,3,False,2017-09-10T11:59:00Z,2017-09-10T11:59:00Z


In [19]:
# Convert state names to title case so that remaining
# capitalization differences are standardized.
df_chargers["state"] = (
    df_chargers["state"]
    .astype(str)
    .str.strip()
    .str.title()
)


In [20]:
# Convert string representations of missing values back
# into actual missing values.
df_chargers["state"] = df_chargers["state"].replace({
    "None": None,
    "Nan": None
})

# Display the cleaned dataset for verification
df_chargers

,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,510840,E-Fill Electric Hotel Food Mania,31.602844,76.924626,"Balh, Mandi",Himachal Pradesh,175021,E-Fill Electric (IN),Operational,True,30.0,1,True,2026-09-30T16:02:00Z,2026-10-03T16:15:00Z
1,510821,Nilkanth EV Charging Station - ChargeJet,23.759334,53.322467,Ahmedabad,Gujarat,382445,(Unknown Operator),Operational,True,160.0,3,True,2026-09-30T04:52:00Z,2026-10-03T04:57:00Z
2,510820,Nilkanth EV Charging Station - ChargeJet,22.979910,72.633820,Ahemdabad,Gujarat,382445,(Unknown Operator),Operational,True,158.0,4,True,2026-09-30T04:24:00Z,2026-10-03T04:26:00Z
3,510563,bhanu prasad hostal,16.258243,80.323226,Guntar,Andhra Pradesh,534350,(Unknown Operator),Operational,True,120.0,1,True,2026-09-26T04:27:00Z,2026-09-29T04:31:00Z
4,510426,SALZER DEVELOPMENT,19.105321,72.875887,NaN,NaN,NaN,Salzer NexCharge (IN),Not Operational,False,7.0,1,False,2026-09-23T16:17:00Z,2026-10-04T04:18:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1974,103604,"International Tech Park, Bangalore",12.985421,77.738513,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04T08:06:00Z,2018-05-11T11:06:00Z
1975,103603,Ather Grid Point - Ascendas Park square mall,12.987219,77.736474,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04T08:01:00Z,2018-05-11T11:06:00Z
1976,96047,Tellus Power,17.413362,78.426597,Hyderabad,Telangana,500034,NaN,Operational,True,7.2,1,False,2017-12-07T10:12:00Z,2017-12-08T08:52:00Z
1977,92772,Tata Power Receiving Station,19.095796,72.926011,Mumbai,NaN,400077,(Business Owner at Location),Operational,True,NaN,3,False,2017-09-10T11:59:00Z,2017-09-10T11:59:00Z


In [21]:
# Count records before removing invalid states
before = df_chargers.shape[0]

# Remove records where the state could not be identified.
# State is important for state-level charging infrastructure analysis, so records without a valid state are excluded.
df_chargers = df_chargers.dropna(subset=["state"])

# Count records after cleaning
after = df_chargers.shape[0]

print(
    f"Dropped {before - after} rows with invalid/missing state. "
    f"Remaining: {after}"
)

Dropped 37 rows with invalid/missing state. Remaining: 1942


In [22]:
# Check the final list of state names after cleaning.
print("\nUnique state values AFTER cleaning:")
print(sorted(df_chargers["state"].unique()))


Unique state values AFTER cleaning:
['Andhra Pradesh', 'Assam', 'Bihar', 'Dadra And Nagar Haveli', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Telangana', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [23]:
# Town is not required for the geographic analysis because latitude and longitude are available.
# Therefore, missing town names are labelled as Unknown.
df_chargers["town"] = (
    df_chargers["town"].fillna("Unknown")
)

In [24]:
# State should already have been cleaned and records with missing states removed above.
# Fill any remaining missing value defensively.
df_chargers["state"] = (
    df_chargers["state"].fillna("Unknown")
)

In [25]:
# Postcode is not required for spatial analysis because geographic coordinates are available.
df_chargers["postcode"] = (
    df_chargers["postcode"].fillna("Unknown")
)

In [26]:
# Operator information is not available for some crowd-sourced charging stations.
df_chargers["operator"] = (
    df_chargers["operator"].fillna("Unknown Operator")
)

In [27]:
# Missing power values are replaced with 0 to retain the station location instead of removing the entire record.
df_chargers["max_power_kw"] = (
    df_chargers["max_power_kw"].fillna(0)
)

In [28]:
# Verify the missing-value treatment
print("Missing values after fill:")
print(df_chargers.isnull().sum())

Missing values after fill:
ID                    0
name                  0
latitude              0
longitude             0
town                  0
state                 0
postcode              0
operator              0
status                0
is_operational        0
max_power_kw          0
total_points          0
has_fast_charging     0
date_created          0
date_last_verified    0
dtype: int64


In [29]:
# Check the minimum and maximum charging power values.
# Extremely large values may indicate data-entry errors.
print(
    "Power range:",
    df_chargers["max_power_kw"].min(),
    "to",
    df_chargers["max_power_kw"].max()
)


Power range: 0.0 to 1000000.0


In [30]:
# Remove records with charging power above 500 kW.
# Such values are treated as unrealistic/outliers for
# this dataset and are excluded from the analysis.
before = df_chargers.shape[0]

df_chargers = df_chargers[
    df_chargers["max_power_kw"] <= 500
]

after = df_chargers.shape[0]

print(
    f"Removed {before - after} rows with unrealistic power values."
)

Removed 2 rows with unrealistic power values.


In [31]:
# Count records before duplicate-location removal
before = df_chargers.shape[0]

# Remove exact duplicate geographic locations.
# Latitude and longitude are used because the analysis
# focuses on the physical location of charging infrastructure.
df_chargers = df_chargers.drop_duplicates(
    subset=["latitude", "longitude"]
)

In [32]:
# Count records after duplicate removal
after = df_chargers.shape[0]

print(
    f"Removed {before - after} duplicate-location stations. "
    f"Remaining: {after}"
)

Removed 97 duplicate-location stations. Remaining: 1843


In [33]:
# India approximately lies between these latitude and longitude ranges. Records outside these ranges are
# checked for possible coordinate errors.
invalid_coords = df_chargers[
    (~df_chargers["latitude"].between(6, 38)) |
    (~df_chargers["longitude"].between(68, 98))
]

print("Invalid coordinates:", invalid_coords.shape[0])

# Display suspicious records for manual inspection
invalid_coords

Invalid coordinates: 2


,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
1,510821,Nilkanth EV Charging Station - ChargeJet,23.759334,53.322467,Ahmedabad,Gujarat,382445,(Unknown Operator),Operational,True,160.0,3,True,2026-09-30T04:52:00Z,2026-10-03T04:57:00Z
1893,301542,Ero G Electric Charging Station,9.357268,11.301994,Erode,Tamil Nadu,638052,Relux Electric (India),Operational,True,44.0,1,True,2024-07-22T07:31:00Z,2024-07-26T06:15:00Z


In [34]:
# ID 301542 has coordinates outside India even though its address indicates Erode, Tamil Nadu.
# Therefore, the record is removed rather than using an incorrect geographic location in spatial analysis.

before = df_chargers.shape[0]

df_chargers = df_chargers[
    df_chargers["ID"] != 301542
]

after = df_chargers.shape[0]

print(
    f"Dropped {before - after} row(s) with invalid coordinates. "
    f"Remaining: {after}"
)

Dropped 1 row(s) with invalid coordinates. Remaining: 1842


In [35]:
# Identify records with zero/negative charging points or unusually large values that require investigation.
invalid_points = df_chargers[
    (df_chargers["total_points"] <= 0) |
    (df_chargers["total_points"] > 50)
]

print("Invalid total_points:", invalid_points.shape[0])

# Display suspicious records for manual inspection
invalid_points

Invalid total_points: 6


,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
323,479330,Electric Vehicle Charging Station,21.016109,81.702900,Unknown,Odisha,Unknown,Unknown Operator,Operational,True,0.0,600,False,2026-02-28T08:08:00Z,2026-03-01T09:24:00Z
1447,309318,Jio-BP Pulse Varthur Hub,12.939790,77.734470,Bengaluru,Karnataka,560087,JIO BP Pulse (India),Operational,True,60.0,136,True,2025-03-06T17:40:00Z,2025-03-07T16:59:00Z
1448,309317,"Jio-BP Pulse Hub, Marathalli",12.951790,77.702070,Bengaluru,Karnataka,560037,JIO BP Pulse (India),Operational,True,60.0,98,True,2025-03-06T17:35:00Z,2025-03-07T16:59:00Z
1507,309186,Jio BP Charging Hub Chikkagubbi,13.087848,77.660148,Bengaluru,Karnataka,562149,JIO BP Pulse (India),Operational,True,60.0,56,True,2025-03-03T17:03:00Z,2025-03-06T17:04:00Z
1609,308922,Blusmart Charging Station (HSR Hub),12.902120,77.641910,Bengaluru,Karnataka,560068,JIO BP Pulse (India),Operational,True,60.0,58,True,2025-02-17T13:06:00Z,2025-03-04T03:24:00Z
1611,308920,JioBP Pulse Charging Hub - Electronic City Pha...,12.864620,77.674860,Bengaluru,Karnataka,560100,JIO BP Pulse (India),Operational,True,60.0,70,True,2025-02-17T12:54:00Z,2025-03-04T03:24:00Z


In [36]:
# Sort records with more than 50 charging points so the largest values can be inspected first.
suspicious = (
    df_chargers[
        df_chargers["total_points"] > 50
    ]
    .sort_values(
        "total_points",
        ascending=False
    )
)

print(suspicious.shape[0], "rows found")

# Display relevant fields for validation
suspicious[[
    "ID",
    "name",
    "town",
    "state",
    "total_points",
    "max_power_kw"
]]

6 rows found


,ID,name,town,state,total_points,max_power_kw
323,479330,Electric Vehicle Charging Station,Unknown,Odisha,600,0.0
1447,309318,Jio-BP Pulse Varthur Hub,Bengaluru,Karnataka,136,60.0
1448,309317,"Jio-BP Pulse Hub, Marathalli",Bengaluru,Karnataka,98,60.0
1611,308920,JioBP Pulse Charging Hub - Electronic City Pha...,Bengaluru,Karnataka,70,60.0
1609,308922,Blusmart Charging Station (HSR Hub),Bengaluru,Karnataka,58,60.0
1507,309186,Jio BP Charging Hub Chikkagubbi,Bengaluru,Karnataka,56,60.0


In [37]:
# ID 479330 reports 600 charging points but has
# 0 kW power and an unknown town.
# This is treated as a clearly erroneous record.
before = df_chargers.shape[0]

df_chargers = df_chargers[
    df_chargers["ID"] != 479330
]

after = df_chargers.shape[0]

print(
    f"Dropped {before - after} clearly invalid record(s). "
    f"Remaining: {after}"
)


# The Bengaluru hub records with high point counts are retained
# because they represent legitimate large charging hubs.
print(
    "New total_points range:",
    df_chargers["total_points"].min(),
    "-",
    df_chargers["total_points"].max()
)

Dropped 1 clearly invalid record(s). Remaining: 1841
New total_points range: 1 - 136


In [38]:
# A station ID should uniquely identify a charging station.
# Duplicate IDs could indicate repeated records.
duplicate_ids = df_chargers[
    df_chargers["ID"].duplicated(keep=False)
]

print("Duplicate IDs:", duplicate_ids.shape[0])

Duplicate IDs: 0


In [39]:
# Check whether the text status agrees with the boolean is_operational field.
inconsistent = df_chargers[
    (
        (df_chargers["status"] == "Operational") &
        (df_chargers["is_operational"] == False)
    )
    |
    (
        (df_chargers["status"] == "Not Operational") &
        (df_chargers["is_operational"] == True)
    )
]


In [40]:
print(
    "Inconsistent status/is_operational rows:",
    inconsistent.shape[0]
)

# Display any inconsistent records
inconsistent[[
    "ID",
    "name",
    "status",
    "is_operational"
]]

Inconsistent status/is_operational rows: 0


,ID,name,status,is_operational


In [41]:
# Round coordinates to four decimal places to identify stations that are extremely close to one another.
# This is only a screening step and does NOT automatically remove the records.
df_chargers["lat_rounded"] = (
    df_chargers["latitude"].round(4)
)

df_chargers["lon_rounded"] = (
    df_chargers["longitude"].round(4)
)

In [42]:
# Identify records sharing the same rounded coordinates
near_duplicates = df_chargers[
    df_chargers.duplicated(
        subset=["lat_rounded", "lon_rounded"],
        keep=False
    )
]
print(
    "Potential near-duplicate stations:",
    near_duplicates.shape[0]
)


Potential near-duplicate stations: 2


In [43]:
# Sort the records before displaying them
near_duplicates = near_duplicates.sort_values(
    ["lat_rounded", "lon_rounded"]
)

near_duplicates[[
    "ID",
    "name",
    "latitude",
    "longitude",
    "town"
]]

,ID,name,latitude,longitude,town
1606,308925,Shivalli Restaurant 1,12.63923,77.14021,Channapatna
1697,308760,Shivalli Chowki Restaurant,12.63923,77.14023,Channapatna


In [44]:

# Compare the complete records before deciding whether
# nearby stations represent duplicates or different operators.
df_chargers[
    df_chargers["ID"].isin([308925, 308760])
]

,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified,lat_rounded,lon_rounded
1606,308925,Shivalli Restaurant 1,12.63923,77.14021,Channapatna,Karnataka,562160,JIO BP Pulse (India),Operational,True,80.0,2,True,2025-02-17T14:18:00Z,2025-03-04T03:24:00Z,12.6392,77.1402
1697,308760,Shivalli Chowki Restaurant,12.63923,77.14023,Channapatna,Karnataka,562160,Zeon Charging,Operational,True,30.0,2,True,2025-02-11T17:11:00Z,2025-03-04T03:29:00Z,12.6392,77.1402


In [45]:
# Both records represent different charging operators
# at almost the same location.
# Therefore, they are retained as separate stations.

df_chargers = df_chargers.drop(
    columns=["lat_rounded", "lon_rounded"]
)

print(
    "Near-duplicate check complete — "
    "no rows dropped, confirmed as distinct chargers."
)

Near-duplicate check complete — no rows dropped, confirmed as distinct chargers.


In [46]:
# Count the number of cleaned charging stations
# available in each state.
print(df_chargers["state"].value_counts())

state
Kerala                    596
Karnataka                 378
Odisha                    284
Tamil Nadu                211
Maharashtra                85
Gujarat                    68
Rajasthan                  47
Delhi                      34
Haryana                    28
Uttar Pradesh              28
Telangana                  17
Uttarakhand                14
Madhya Pradesh             13
West Bengal                 9
Andhra Pradesh              8
Goa                         6
Himachal Pradesh            3
Dadra And Nagar Haveli      3
Assam                       2
Jharkhand                   2
Punjab                      2
Jammu And Kashmir           1
Puducherry                  1
Bihar                       1
Name: count, dtype: int64


In [47]:
# Convert API date strings into proper datetime values.
# errors="coerce" converts invalid date values into NaT
# instead of causing the notebook to fail.
df_chargers["date_created"] = pd.to_datetime(
    df_chargers["date_created"],
    errors="coerce"
)

df_chargers["date_last_verified"] = pd.to_datetime(
    df_chargers["date_last_verified"],
    errors="coerce"
)


# Confirm that the date columns were converted successfully
print(
    df_chargers[
        ["date_created", "date_last_verified"]
    ].dtypes
)

date_created          datetime64[us, UTC]
date_last_verified    datetime64[us, UTC]
dtype: object


In [48]:
df_chargers

,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,510840,E-Fill Electric Hotel Food Mania,31.602844,76.924626,"Balh, Mandi",Himachal Pradesh,175021,E-Fill Electric (IN),Operational,True,30.0,1,True,2026-09-30 16:02:00+00:00,2026-10-03 16:15:00+00:00
1,510821,Nilkanth EV Charging Station - ChargeJet,23.759334,53.322467,Ahmedabad,Gujarat,382445,(Unknown Operator),Operational,True,160.0,3,True,2026-09-30 04:52:00+00:00,2026-10-03 04:57:00+00:00
2,510820,Nilkanth EV Charging Station - ChargeJet,22.979910,72.633820,Ahemdabad,Gujarat,382445,(Unknown Operator),Operational,True,158.0,4,True,2026-09-30 04:24:00+00:00,2026-10-03 04:26:00+00:00
3,510563,bhanu prasad hostal,16.258243,80.323226,Guntar,Andhra Pradesh,534350,(Unknown Operator),Operational,True,120.0,1,True,2026-09-26 04:27:00+00:00,2026-09-29 04:31:00+00:00
11,510152,Subhash Super Fast EV Charging Station | 90kw ...,11.298295,75.861697,Kozhikode,Kerala,673571,Chargezone (India),Operational,True,120.0,6,True,2026-09-19 12:19:00+00:00,2026-09-22 12:23:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1973,103606,Ather Grid Point - Moto Store & Cafe,12.987554,77.620871,Bangalore,Karnataka,560042,(Unknown Operator),Operational,True,3.0,1,False,2018-05-04 08:14:00+00:00,2018-05-11 11:06:00+00:00
1974,103604,"International Tech Park, Bangalore",12.985421,77.738513,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04 08:06:00+00:00,2018-05-11 11:06:00+00:00
1975,103603,Ather Grid Point - Ascendas Park square mall,12.987219,77.736474,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04 08:01:00+00:00,2018-05-11 11:06:00+00:00
1976,96047,Tellus Power,17.413362,78.426597,Hyderabad,Telangana,500034,Unknown Operator,Operational,True,7.2,1,False,2017-12-07 10:12:00+00:00,2017-12-08 08:52:00+00:00


In [49]:
#Check unique appointment status values- unique()
df_chargers["town"].unique()

<ArrowStringArray>
[                    'Balh, Mandi',                       'Ahmedabad',
                       'Ahemdabad',                          'Guntar',
                       'Kozhikode',                           'Mandi',
                      'MIRYALGUDA',                           'Ashta',
                         'Unknown',                        'Durgapur',
 ...
                        'Vandalur',                 'Irungattukottai',
            ' Chikkanayakanahalli',                'Mukundgarh Mandi',
                       'Vaigundam', 'Chennai City Corporation Limits',
                        'Guwahati',                       'Darbhanga',
                        'Clappana',                          'Nagpur']
Length: 759, dtype: str

In [50]:
# We intentionally do not manually map every town name because town-level names are not the main variable used in the highway corridor analysis.
df_chargers["town"] = (
    df_chargers["town"]
    .str.strip()
    .str.title()
)

# Check how many unique town values remain
print(
    "Unique towns after basic cleanup:",
    df_chargers["town"].nunique()
)

Unique towns after basic cleanup: 738


In [51]:
df_chargers["operator"].unique()

<ArrowStringArray>
[             'E-Fill Electric (IN)',                '(Unknown Operator)',
                'Chargezone (India)',             'Charge_iN by Mahindra',
                    'ChargeMod (IN)',      '(Business Owner at Location)',
                        'Tata Power',                       'Statiq (IN)',
                     'Nikol EV (IN)',                     'Zeon Charging',
               'Shell Recharge (IN)',              'JIO BP Pulse (India)',
                 'Adani Gas-EV (TR)',                        'GO EC (IN)',
                   'Bolt.Earth (IN)',                  'Unknown Operator',
                  'EarthtronEV (IN)',                  'eDrive BPCL (IN)',
                         'Tryk (IN)',                       'Xobolt (IN)',
                      'Volttic (IN)',               'Hydra Charging (IN)',
            'Relux Electric (India)',                         'EESL (IN)',
                           'Ecoplug', 'GE WattStation (No longer active)',
      

In [52]:
# Merge two labels that represent the same missing/
# unknown operator category.
df_chargers["operator"] = (
    df_chargers["operator"].replace({
        "(Unknown Operator)": "Unknown Operator"
    })
)

print(
    "Unique operators after fix:",
    df_chargers["operator"].nunique()
)

Unique operators after fix: 35


In [53]:
df_chargers["status"].unique()

<ArrowStringArray>
[               'Operational',            'Not Operational',
    'Temporarily Unavailable', 'Partly Operational (Mixed)',
    'Planned For Future Date']
Length: 5, dtype: str

In [54]:
# Check the boolean operational values.
df_chargers["is_operational"].unique()

array([ True, False])

In [55]:
# Check the fast-charging indicator values.
df_chargers["has_fast_charging"].unique()

array([ True, False])

In [56]:
print("Final cleaned shape:", df_chargers.shape)

Final cleaned shape: (1841, 15)


In [57]:
df_chargers= df_chargers.reset_index(drop = True)

In [58]:
df_chargers.to_csv("../data/Cleaned datasets/Ev_chargers_cleaned.csv", index=False)
print("Saved successfully!")

Saved successfully!


In [59]:
df_chargers

,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,510840,E-Fill Electric Hotel Food Mania,31.602844,76.924626,"Balh, Mandi",Himachal Pradesh,175021,E-Fill Electric (IN),Operational,True,30.0,1,True,2026-09-30 16:02:00+00:00,2026-10-03 16:15:00+00:00
1,510821,Nilkanth EV Charging Station - ChargeJet,23.759334,53.322467,Ahmedabad,Gujarat,382445,Unknown Operator,Operational,True,160.0,3,True,2026-09-30 04:52:00+00:00,2026-10-03 04:57:00+00:00
2,510820,Nilkanth EV Charging Station - ChargeJet,22.979910,72.633820,Ahemdabad,Gujarat,382445,Unknown Operator,Operational,True,158.0,4,True,2026-09-30 04:24:00+00:00,2026-10-03 04:26:00+00:00
3,510563,bhanu prasad hostal,16.258243,80.323226,Guntar,Andhra Pradesh,534350,Unknown Operator,Operational,True,120.0,1,True,2026-09-26 04:27:00+00:00,2026-09-29 04:31:00+00:00
4,510152,Subhash Super Fast EV Charging Station | 90kw ...,11.298295,75.861697,Kozhikode,Kerala,673571,Chargezone (India),Operational,True,120.0,6,True,2026-09-19 12:19:00+00:00,2026-09-22 12:23:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1836,103606,Ather Grid Point - Moto Store & Cafe,12.987554,77.620871,Bangalore,Karnataka,560042,Unknown Operator,Operational,True,3.0,1,False,2018-05-04 08:14:00+00:00,2018-05-11 11:06:00+00:00
1837,103604,"International Tech Park, Bangalore",12.985421,77.738513,Bangalore,Karnataka,560066,Unknown Operator,Operational,True,3.0,2,False,2018-05-04 08:06:00+00:00,2018-05-11 11:06:00+00:00
1838,103603,Ather Grid Point - Ascendas Park square mall,12.987219,77.736474,Bangalore,Karnataka,560066,Unknown Operator,Operational,True,3.0,2,False,2018-05-04 08:01:00+00:00,2018-05-11 11:06:00+00:00
1839,96047,Tellus Power,17.413362,78.426597,Hyderabad,Telangana,500034,Unknown Operator,Operational,True,7.2,1,False,2017-12-07 10:12:00+00:00,2017-12-08 08:52:00+00:00
